# TSMOM risk-budget diagnostic: current cluster-cap vs. ERC vs. HRP

Wraps `scripts/tsmom_risk_budget_diagnostic.py` -- see that file's module
docstring for the full context (motivated by
`research/cta-layer-separation-risk-budgeting.md`), the library
deviations from the original brief (riskfolio-lib uninstalled after it
broke numpy/pandas project-wide; ERC is a small scipy solver instead;
HRP needs a one-line scipy>=1.18 compatibility shim), and exactly what
each returned field means.

Requires a live IB connection (TWS/IB Gateway running) -- this notebook
doesn't change that, it just calls `main()` directly instead of going
through a real command line, and gives you the returned artifacts
(correlation matrix, weights, etc.) to plot/inspect inline.

In [ ]:
import sys
import logging
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'nbs' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

import hvplot.polars   # registers .hvplot on polars DataFrames/Series
import polars as pl

from scripts.tsmom_risk_budget_diagnostic import main

## Params

Same flags as the CLI (`python -m scripts.tsmom_risk_budget_diagnostic --help`
for the full list with descriptions) -- edit this list and re-run the next
cell to try different account sizes, EWM halflives, instrument subsets, etc.
`--account-equity` is the only required one.

In [ ]:
# Instruments: defaults to the full KNOWN_INSTRUMENTS universe.
# Uncomment and edit to narrow the universe.
# INSTRUMENTS = 'MES,MNQ,ES,NQ'

account_equity = 80_000
halflife = 60          # EWM covariance halflife (days)
min_synced_rows = 252  # require >= 1y of common history across every instrument

HOST      = '127.0.0.1'
PORT      = 7496
CLIENT_ID = 21

argv = [
    '--account-equity', str(account_equity),
    '--halflife', str(halflife),
    '--min-synced-rows', str(min_synced_rows),
    '--host', HOST,
    '--port', str(PORT),
    '--client-id', str(CLIENT_ID),
    '--no-save',  # comment out to also write CSVs to results/
]
if 'INSTRUMENTS' in dir():
    argv += ['--instruments', INSTRUMENTS]

In [ ]:
# Connects to IB, fetches 3y of continuous front-month bars for every
# instrument, computes ERC/HRP/current-system weights, and returns every
# intermediate artifact -- same console output as the CLI, plus the dict.
results = main(argv)
results.keys()

In [ ]:
report = results['report']
summary = results['summary']
cm = results['corr']  # correlation matrix, sorted by hand-assigned cluster

report

In [ ]:
summary

## Correlation matrix

Sorted by the live system's hand-assigned cluster, then symbol -- if the
clusters are picking up a real factor, same-cluster blocks along the
diagonal should visibly run hotter than cross-cluster cells.

In [ ]:
# Correlation matrix heatmap -- polars at the plot boundary (cm is pandas from correlation_view).
cm_long = (
    pl.from_pandas(cm.reset_index().rename(columns={'index': 'instrument'}))
    .unpivot(index='instrument', variable_name='col', value_name='corr')
)
cm_long.hvplot.heatmap(
    x='col', y='instrument', C='corr',
    cmap='coolwarm', clim=(-1, 1), colorbar=True,
    width=600, height=550,
    title=f'EWM correlation (halflife={halflife}d), sorted by hand-assigned cluster',
)

## Weight comparison: current cluster-cap vs. ERC vs. HRP

In [ ]:
# Weight comparison bar chart -- convert report (pandas) to polars at the plot boundary.
(
    pl.from_pandas(
        report[['current_weight', 'erc_weight', 'hrp_weight']]
        .reset_index()
        .rename(columns={'index': 'symbol'})
    )
    .unpivot(index='symbol', on=['current_weight', 'erc_weight', 'hrp_weight'],
             variable_name='method', value_name='weight')
    .hvplot.bar(
        x='symbol', y='weight', by='method',
        ylabel='Weight (fraction of total |risk|)',
        title='Risk-budget weight by instrument: current system vs. ERC vs. HRP',
        width=900, height=400, rot=45,
    )
)

In [ ]:
# Divergence from current weights -- where would ERC/HRP size up or down?
(
    pl.from_pandas(
        report[['erc_minus_current', 'hrp_minus_current']]
        .reset_index()
        .rename(columns={'index': 'symbol'})
    )
    .unpivot(index='symbol', on=['erc_minus_current', 'hrp_minus_current'],
             variable_name='method', value_name='divergence')
    .hvplot.bar(
        x='symbol', y='divergence', by='method',
        ylabel='Divergence from current weight',
        title='Where would ERC/HRP size up or down relative to today?',
        width=900, height=400, rot=45,
    )
)